# AI Photo Enhancement
**WindowSeat Reflection Removal + NAFNet Denoise/Deblur + Real-ESRGAN Upscale**

This notebook runs the full application with a web UI accessible via a public URL.

**Requirements:**
- GPU runtime (T4 free tier works, A100/L4 recommended)
- HuggingFace token with access to `Qwen/Qwen-Image-Edit-2509`

---

## 1. Check GPU

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB\" if torch.cuda.is_available() else \"\")

SyntaxError: unexpected character after line continuation character (1674539126.py, line 5)

## 2. Clone Repository & Install Dependencies

In [ ]:
import os
os.chdir('/content')

# Clone the repo
if not os.path.exists('AI-photo-enhancement'):
    !git clone https://github.com/buianhhuy96/AI-photo-enhancement.git
else:
    !cd AI-photo-enhancement && git pull

os.chdir('/content/AI-photo-enhancement')
print("\n✓ Repository ready")

In [ ]:
# Install Python dependencies
!pip install -q fastapi uvicorn[standard] python-multipart pydantic \
    torch torchvision diffusers transformers accelerate \
    safetensors huggingface_hub peft bitsandbytes \
    imageio numpy Pillow tqdm basicsr realesrgan

print("\n✓ Python packages installed")

## 3. Build Frontend

In [ ]:
# Install Node.js and build the React frontend
!curl -fsSL https://deb.nodesource.com/setup_20.x | sudo -E bash - > /dev/null 2>&1
!sudo apt-get install -y nodejs > /dev/null 2>&1
!node --version

os.chdir('/content/AI-photo-enhancement/web')
!npm install --legacy-peer-deps 2>&1 | tail -3
!npx vite build 2>&1 | tail -5
os.chdir('/content/AI-photo-enhancement')

print("\n✓ Frontend built to web/dist/")

## 4. Set HuggingFace Token & Download Models

You need a HuggingFace token with access to `Qwen/Qwen-Image-Edit-2509`.
Get one at https://huggingface.co/settings/tokens

In [ ]:
from google.colab import userdata
import os

# Option 1: Use Colab Secrets (recommended)
# Add a secret named 'HF_TOKEN' in the key icon on the left sidebar
try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print("✓ Using HF_TOKEN from Colab Secrets")
except Exception:
    # Option 2: Paste your token here
    os.environ['HF_TOKEN'] = ''  # <-- paste your token here if not using Secrets
    if not os.environ['HF_TOKEN']:
        print("⚠ No token set! Add HF_TOKEN to Colab Secrets or paste it above.")
    else:
        print("✓ Using pasted HF_TOKEN")

# Login to HuggingFace
!huggingface-cli login --token $HF_TOKEN 2>&1 | tail -2

In [ ]:
# Download model weights (~10GB total)
os.chdir('/content/AI-photo-enhancement')
!python3 download_models.py

## 5. Start Server with Public URL

This starts the FastAPI server and creates a public tunnel using cloudflared.
Click the URL that appears to open the web UI in your browser.

In [ ]:
# Install cloudflared for tunneling
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
!rm cloudflared-linux-amd64.deb
print("✓ cloudflared installed")

In [ ]:
import subprocess
import time
import threading
import re
import os

os.chdir('/content/AI-photo-enhancement')

# Start the FastAPI server in background
server_proc = subprocess.Popen(
    ['python3', 'serve.py', '--port', '8000'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

# Wait for server to start
print("Starting server...")
time.sleep(5)

# Start cloudflared tunnel
tunnel_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:8000'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

# Extract the public URL
print("Creating tunnel...")
url_found = False
for _ in range(30):
    line = tunnel_proc.stdout.readline()
    if line:
        match = re.search(r'(https://[\w-]+\.trycloudflare\.com)', line)
        if match:
            public_url = match.group(1)
            print(f"\n{'='*60}")
            print(f"  ✓ App is live!")
            print(f"  Open this URL in your browser:")
            print(f"  {public_url}")
            print(f"{'='*60}\n")
            url_found = True
            break

if not url_found:
    print("⚠ Could not detect tunnel URL. Check output below.")

# Keep printing server logs
def print_logs():
    while True:
        line = server_proc.stdout.readline()
        if not line:
            break
        print(f"[server] {line.strip()}")

log_thread = threading.Thread(target=print_logs, daemon=True)
log_thread.start()

print("Server is running. Keep this cell active.")
print("To stop: Runtime → Interrupt execution")

---
## Alternative: Mock Mode (no GPU needed, for UI testing only)

If you just want to test the UI without AI processing:

In [ ]:
# # Uncomment to run in mock mode (no AI, just UI testing)
# import subprocess, time, re
# os.chdir('/content/AI-photo-enhancement')
# server_proc = subprocess.Popen(
#     ['python3', 'serve.py', '--mock', '--port', '8000'],
#     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
# )
# time.sleep(3)
# tunnel_proc = subprocess.Popen(
#     ['cloudflared', 'tunnel', '--url', 'http://localhost:8000'],
#     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
# )
# for _ in range(30):
#     line = tunnel_proc.stdout.readline()
#     if line:
#         match = re.search(r'(https://[\w-]+\.trycloudflare\.com)', line)
#         if match:
#             print(f"\n✓ Mock UI: {match.group(1)}\n")
#             break